## Colab Runtime Setup
Before running the rest of the notebook:
1. Go to **Runtime > Change runtime type** and select a **GPU** (T4 is fine).
2. Run the two setup cells below to confirm the GPU is attached and mount Google Drive so checkpoints (and the downloaded MNIST data) persist across sessions.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/ECE1508_A5_Q2'
os.makedirs(save_dir, exist_ok=True)
os.chdir(save_dir)
print(f"Working directory set to: {os.getcwd()}")

# ECE1508: Deep Generative Models -- Summer 2026
## Assignment 5: Diffusion Models
## Question 2: Diffusion Score Model

In this assignment we try to build the simplest possible diffusion model and train it on the subset of MNIST dataset that include only digit 8. The main purpose is to review what we have learned about diffusion and its connection to data generation. Our model consists of a simple score network which is trained using the diffusion score matching idea: _add noise to samples and estimate their score using the noise term._ We train this network on MNIST and then use it to generate samples by a simple reverse SDE.

### Loading Modules

In [1]:
import torch, torchvision
from torch.utils.data import TensorDataset, DataLoader, Subset, random_split
import torch.nn as nn
import torch.nn.functional as F

import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST, MNIST
import matplotlib.pyplot as plt
from tqdm import tqdm


import numpy as np
import random
import math

# select a device that we want to shift the process on: cuda, cpu, mps
if torch.backends.mps.is_built():
    device = "mps"
elif torch.backends.cuda.is_built():
    device = "cuda"
else:
    device = "cpu"

### Building the Score Net
Our score net is a basic CNN with 8 layers. The number of channels change in the following order
$$
1 (\text{MNIST}) \mapsto 32 \mapsto 64 \mapsto 128 \mapsto 256 \mapsto 128 \mapsto 64 \mapsto 32 \mapsto 1
$$
each layer uses batch normalization. We activate each layer by $\mathrm{SiLU}$ function.

To include the noise standard deviation $\sigma$, we use the following trick: we first embed the noise standard deviation via the following positional encoding:
1. choose an even embedding dimension $d$ 
2. for $i = 0,1,\ldots, d/2-1$ find the following frequencies
$$
\omega_i = 10000^{- \frac{i}{d/2-1} }
$$
3. compute the following embedding vector 
$$
\bold{e} (\sigma) = [ \sin (\omega_0 \log \sigma ), \sin (\omega_1 \log \sigma ), \ldots, \sin (\omega_{d/2-1} \log \sigma ), 
\cos (\omega_0 \log \sigma ), \cos (\omega_1\log \sigma ), \ldots, \cos (\omega_{d/2-1} \log \sigma )
]
$$

Then, in each convolutional layer, we add the impact of noise standard deviation by projecting its embedding $\bold{e} (\sigma)$ via a linear layer and adding it to the output of convolution _after batch normalization_ and _before activation._ The following code implement this score net.

In [ ]:
class ScoreNet(nn.Module):
    def __init__(self, embed_dim=32):
        '''
        embed_dim = size of noise embedding
        '''
        super().__init__()
        
        self.embed_dim = embed_dim
        
        ## COMPLETE ##
        channels = [1, 32, 64, 128, 256, 128, 64, 32, 1]
        self.convs = nn.ModuleList([
            nn.Conv2d(channels[i], channels[i+1], kernel_size=3, padding=1)
            for i in range(len(channels)-1)
        ])

        self.norms = nn.ModuleList([
            nn.BatchNorm2d(channels[i+1]) for i in range(len(channels)-1)
        ])

        # Noise conditioning layers
        self.noise_proj = nn.ModuleList([
            nn.Linear(embed_dim, channels[i+1]) for i in range(len(channels)-1)
        ])
        
        ## COMPLETE ##
        self.act = nn.SiLU()
        
        
        
    def _get_embedding(self, log_sigma, embedding_dim):
        half_dim = embedding_dim // 2
        
        # compute frequencies omega
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half_dim, device=log_sigma.device).float() / (half_dim - 1)
        )

        ## COMPLETE ##
        args = log_sigma.view(-1,1) * freqs.view(1,-1)

        # get the positional encoding
        ## COMPLETE ##
        embedding = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return embedding

       
        
    def forward(self, x, sigma):
        B, C, H, W = x.shape
        
        # embed the noise
        ## COMPLETE ##
        log_sigma = torch.log(sigma).view(B)
        emb = self._get_embedding(log_sigma, self.embed_dim) # (B, embed_dim)
        
        # pass through conv and normalize
        ## COMPLETE ##
        h = x
        n_layers = len(self.convs)

        # add noise conditioning
        ## COMPLETE ##
        for idx, (conv, norm, proj) in enumerate(zip(self.convs, self.norms, self.noise_proj)):
            h = conv(h)
            if idx < n_layers - 1:
                h = norm(h)
                h = h + proj(emb).view(B, -1, 1, 1)
                h = self.act(h)
        return h 

### Defining DSM Loss
We next define the loss of diffusion score matching. Here, we consider direct estimation of the score given by the noise term.

In [ ]:
def dsm_loss(score_net, x, device, sigma_min=1e-6, sigma_max=1.0):
    '''
    sigma_min and sigma_max are the range of sigma we train the score net on
    '''
    B = x.shape[0]
    
    # Sample random noise levels
    log_sigma = torch.rand(B, 1, 1, 1, device=device) * \
        (np.log(sigma_max) - np.log(sigma_min)) + np.log(sigma_min)
    sigma = torch.exp(log_sigma)
    
    # Add noise
    noise = torch.randn_like(x)
    # x_noisy = ## COMPLETE ##
    x_noisy = x + sigma * noise 
    
    # Predict score by Score Net
    # score_pred = ## COMPLETE ##
    score_pred = score_net(x_noisy, sigma)
    
    # Find the score estimate from noise 
    # target = ## COMPLETE ##
    target = -noise / sigma
   
    # for training stability, we scale thr loss with sigma^2
    # weights = ## COMPLETE ##
    # loss = ## COMPLETE ##
    weights = sigma ** 2
    loss = torch.mean(weights * (score_pred - target) ** 2)
    
    return loss

### Sampling Diffusion Process
We now sample the diffusion trajectory. In this trajectory, we consider the forward SDE as
$$
\mathrm{d} x_t = -\frac{1}{2} \beta_t \mathrm{d} t + \sqrt{\beta_t} \mathrm{d} B_t
$$
where here $\beta_t$ is a function of time. We consider a simple linear function:
$$
\beta_t = \beta_{\min} + (\beta_{\max}- \beta_{\min}) t
$$
for some minimum and maximum choice of $\beta$. 

Note that we use the __reverse SDE__ to sample from data distribution. In our implementation, we approximate the eduation by breaking the time interval $t\in [10^{-6}, 1]$ into 1000 time steps, i.e., $\mathrm{d} t \approx 0.001$. You can increase the accuracy as you wish.


### Question: _Write down the reverse SDE._
$$\mathrm{d}x_t = \left[-\tfrac12\beta_t x_t - \beta_t \nabla_x \log p_t(x_t)\right]\mathrm{d}t + \sqrt{\beta_t},\mathrm{d}\bar B_t$$

__Hint:__ When you build the reverse SDE pay attention to the sign of time difference $\mathrm{d} t$ when multiplied by drift term. 

In [ ]:
# write the beta function
def beta_schedule(t, beta_min=0.1, beta_max=10.0):
    ## COMPLETE ##
    return beta_min + (beta_max - beta_min) * t


# write sampling function
@torch.no_grad()
def sample_from_model(score_net, shape, device, num_steps=1000):
    score_net.eval()

    # sample noise
    x = torch.randn(shape, device=device)

    # specify time steps backward 1 -> 0
    t_steps = torch.linspace(1.0, 1e-6, num_steps, device=device)

    # start the trajectory
    for i in range(num_steps - 1):
        # time steps and dt
        ## COMPLETE ##
        t, t_next = t_steps[i], t_steps[i+1]
        dt = t_next - t

        # find beta_t
        # ## COMPLETE ##
        beta_t = beta_schedule(t)
        sigma = t

        score = score_net(x, sigma.view(1, 1, 1, 1).expand(shape[0], 1, 1, 1))

        drift = -0.5 * beta_t * x - beta_t * score
        diffusion = torch.sqrt(beta_t)

        z = torch.randn_like(x)
        x = x + drift * dt + diffusion * torch.sqrt(-dt) * z

        if (i+1) % 200 == 0 or i==0:
            print(f" Step {i+1}/{num_steps}, t={t:.4f}, noise SD={sigma:.2f}")

    return x


### Training DSM 
We now train our score model on digit 8 images in MNIST. Let us first write the train function.

In [ ]:
def train_score_model(model, train_loader, device, num_epochs = 100):
    
    # Optimizer: for simplicity included in the function
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-6)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)
    

    # Let's visualize the output every couple of epochs
    sample_interval = 10
    losses = []
    
    # training loop
    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []
        
        # for visualization
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        
        # batch computations visualized
        for _, (data, _) in enumerate(pbar):
            data = data.to(device)
            
              
            optimizer.zero_grad()
            
            # Compute DSM loss
            # loss = ## COMPLETE ##
            loss = dsm_loss(model, data, device)
            loss.backward()
            
            # Add Gradient clipping for stability
            ## COMPLETE ##
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
            
            optimizer.step()
            epoch_losses.append(loss.item())
            
            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # schedule the learning rate
        scheduler.step()
        
        # Log epoch results
        avg_loss = np.mean(epoch_losses)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}, LR = {scheduler.get_last_lr()[0]:.6f}')
        
        # Generate samples every couple of epochs
        if (epoch + 1) % sample_interval == 0:
            print("Generating samples...")
            model.eval()
            
            # sample a batch of 6 images
            samples = sample_from_model(
                model, 
                (6, 1, 28, 28), 
                device,
                num_steps=500
            )
            
            # Normalize samples based on data range
            samples = torch.clamp(samples, -1, 1)
            
            # Plot samples
            fig, axes = plt.subplots(1, 6, figsize=(12,2))
            ## COMPLETE ##
            for i in range(6):
                axes[i].imshow(samples[i, 0].cpu().numpy(), cmap='gray')
                axes[i].axis('off')
            plt.show()
        
        # Save checkpoint if you like
        if (epoch + 1) % 50 == 0:
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'losses': losses
            }
            torch.save(checkpoint, f'dsm_model_epoch_{epoch+1}.pt')
            print(f"Checkpoint saved: dsm_model_epoch_{epoch+1}.pt")
    
    # Final training curve
    plt.figure(figsize=(10, 6))
    ## COMPLETE ##
    plt.plot(losses)
    plt.title('DSM Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.show()

    return model, losses

### Train on MNIST
We now train it on MNIST and look at the result. First we instantiate the model and prepare dataloader.

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
full_dataset = MNIST('data', train=True, download=True, transform=transform)

# Find indices of digit 8
digit_8_indices = [i for i, (_, target) in enumerate(full_dataset) if target == 8]

# Create dataset
train_dataset = Subset(full_dataset, digit_8_indices)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)

# model = ## COMPLETE ## Set embd_dim = 64
model = ScoreNet(embed_dim=64).to(device)
num_epochs = 100

and now we train

In [ ]:
train_score_model(model, train_loader, device, num_epochs = 100)

### Question: _What do you observe as the training progress?_
COMPLETE